# Notebook 02 — Dataset PyTorch, DataLoaders y Augmentation
## Detección de Muelas del Juicio Impactadas con YOLOv8

**Materia:** Redes Neuronales  
**Docente:** Ing. Pablo Marinozi  
**Repositorio:** https://github.com/JulianOliveraBalls/wisdomscan

---

Este notebook asume que el Notebook 01 fue ejecutado y los datasets están generados en `data/processed/`. Si no es así, ejecutar primero `01_dataset_preparation.ipynb` o el script de descarga:


### Secciones

| Sección | Contenido |
|---------|----------|
| 0 | Setup del entorno |
| a | Carga y organización del dataset — clase `DentexDataset` y `DentexDetectionDataset` |
| b | Particionado — train/val/test con distribución por clase |
| c | Preprocesamiento — resize, normalización, manejo de escala de grises |
| d | Data augmentation — 6 estrategias, visualización, confirmación de separación train/val/test |
| e | Verificación final — batch visual con boxes desnormalizadas, dimensiones y rangos |


---

## Sección 0 — Setup del entorno


In [ ]:
import subprocess, sys, os, json, random, warnings, shutil
from pathlib import Path
from collections import Counter, defaultdict
warnings.filterwarnings('ignore')

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *pkgs, '-q'])

try:
    import ultralytics, sklearn
except ImportError:
    pip_install('ultralytics', 'scikit-learn')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF

# ── Utilidades ────────────────────────────────────────────────────────────
def log(msg, level='INFO'):
    icons = {'INFO':'[INFO]','OK':'[OK]  ','WARN':'[WARN]','ERR':'[ERR] ','DATA':'[DATA]'}
    print(f'{icons.get(level, "[INFO]")} {msg}')

def is_real_file(p):
    try: return Path(p).resolve().exists() and Path(p).stat().st_size > 0
    except: return False

# ── Detección Colab / local ───────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = os.path.exists('/content') and not os.path.exists('C:/Users')
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive; drive.mount('/drive')
    REPO_ROOT = Path('/content/wisdomscan')
    if not REPO_ROOT.exists():
        subprocess.run(['git', 'clone',
            'https://github.com/JulianOliveraBalls/wisdomscan.git',
            str(REPO_ROOT)], check=True)
    DRIVE_DIR = Path('/drive/MyDrive/dentex_runs')
else:
    _nb = Path(globals().get('__vsc_ipynb_file__',
               globals().get('__file__', str(Path.cwd()/'dev'/'notebook.ipynb'))))
    REPO_ROOT = _nb.parent.parent
    DRIVE_DIR = REPO_ROOT / 'dev'

DATA_DIR    = REPO_ROOT / 'data'
PROCESSED   = DATA_DIR / 'processed'
YOLO_MERGED = PROCESSED / 'yolo_merged'
YOLO_G8b    = PROCESSED / 'yolo_g8b_small'
OUTPUTS_DIR = DATA_DIR / 'outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

SEED       = 42
BATCH_SIZE = 8
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

log(f'REPO_ROOT:  {REPO_ROOT}', 'DATA')
log(f'IN_COLAB:   {IN_COLAB}  |  device: {device}  |  SEED: {SEED}', 'OK')

# ── Verificar que el dataset del notebook 01 existe ──────────────────────
for name, path in [('yolo_merged', YOLO_MERGED), ('yolo_g8b_small', YOLO_G8b)]:
    if path.exists() and (path/'dataset.yaml').exists():
        n = sum(len(list((path/'images'/s).glob('*.*'))) for s in ['train','val','test'])
        log(f'{name}: OK ({n} imgs totales)', 'OK')
    else:
        log(f'{name}: NO ENCONTRADO — ejecutar notebook 01 primero', 'ERR')


---

## a) Carga y organización del dataset

El formato de los datasets (estructura de carpetas YOLO, archivos `.txt` de labels, `dataset.yaml`) está documentado en el Notebook 01. Acá se implementa la capa PyTorch que los consume.

### ¿Por qué no `ImageFolder`?

`ImageFolder` asume **una imagen = una clase** (clasificación). En detección, cada imagen puede tener múltiples bounding boxes. Se implementa `DentexDataset` que lee el par `(imagen, .txt de labels)` y devuelve `(Tensor, Tensor[N,5])`.

Se implementan dos variantes:
- `DentexDataset` — imagen normalizada + boxes YOLO `[cls, cx, cy, w, h]` para entrenamiento y verificación
- `DentexDetectionDataset` — imagen PIL + boxes absolutas en píxeles para visualización directa


In [ ]:
class DentexDataset(Dataset):
    """
    Dataset YOLO de radiografias panoramicas para PyTorch.

    Devuelve:
        img   : Tensor [3, H, W] normalizado con la transform indicada
        boxes : Tensor [N, 5]  donde cada fila es [cls, cx, cy, w, h] normalizado
                (N puede ser 0 si la imagen no tiene anotaciones)

    Nota sobre el uso con YOLOv8:
        YOLOv8 tiene su propio pipeline interno de datos que lee directamente
        los archivos YOLO y aplica augmentation sincronizada con los boxes.
        Esta clase cumple los requisitos de la catedra (Dataset PyTorch) y se usa
        para verificacion visual. El entrenamiento real usa model.train(data=yaml).
    """
    CLASS_NAMES  = {0: 'impacted'}
    CLASS_COLORS = {0: '#FF4444'}

    def __init__(self, split: str, yolo_dir: Path, transform=None):
        self.split     = split
        self.yolo_dir  = Path(yolo_dir)
        self.transform = transform
        img_dir = self.yolo_dir / 'images' / split
        lbl_dir = self.yolo_dir / 'labels' / split
        self.samples = [
            (p, lbl_dir / (p.stem + '.txt'))
            for p in sorted(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
            if (lbl_dir / (p.stem + '.txt')).exists()
        ]
        self.class_counts = Counter()
        for _, lp in self.samples:
            for line in lp.read_text().strip().split('\n'):
                if line.strip():
                    self.class_counts[int(line.split()[0])] += 1

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        img_path, lbl_path = self.samples[idx]
        real_path = Path(os.path.realpath(str(img_path)))
        pil = Image.open(real_path).convert('RGB')
        if self.transform:
            img_tensor = self.transform(pil)
        else:
            img_tensor = TF.to_tensor(pil)
        boxes = []
        for line in lbl_path.read_text().strip().split('\n'):
            if line.strip():
                parts = line.split()
                boxes.append([int(parts[0]), float(parts[1]),
                               float(parts[2]), float(parts[3]), float(parts[4])])
        boxes_t = torch.tensor(boxes, dtype=torch.float32) if boxes \
                  else torch.zeros((0, 5), dtype=torch.float32)
        return img_tensor, boxes_t


class DentexDetectionDataset(Dataset):
    """
    Variante para visualizacion: devuelve imagen PIL sin normalizar
    y boxes en coordenadas absolutas (pixeles), adecuadas para dibujar directamente.
    Solo se usa para inspeccion visual — no para entrenamiento.
    """
    def __init__(self, split: str, yolo_dir: Path):
        self.split    = split
        self.yolo_dir = Path(yolo_dir)
        img_dir = self.yolo_dir / 'images' / split
        lbl_dir = self.yolo_dir / 'labels' / split
        self.samples = [
            (p, lbl_dir / (p.stem + '.txt'))
            for p in sorted(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
            if (lbl_dir / (p.stem + '.txt')).exists()
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, lbl_path = self.samples[idx]
        real_path = Path(os.path.realpath(str(img_path)))
        pil = Image.open(real_path).convert('RGB')
        W, H = pil.size
        boxes_abs = []
        for line in lbl_path.read_text().strip().split('\n'):
            if line.strip():
                cls, cx, cy, bw, bh = map(float, line.split())
                x1 = (cx - bw/2) * W; y1 = (cy - bh/2) * H
                x2 = (cx + bw/2) * W; y2 = (cy + bh/2) * H
                boxes_abs.append([int(cls), x1, y1, x2, y2])
        return pil, boxes_abs, img_path.name


def collate_fn(batch):
    """Collate para deteccion: devuelve lista de tensores de boxes (N puede variar)."""
    images = torch.stack([item[0] for item in batch])
    boxes  = [item[1] for item in batch]   # lista de tensores [Ni, 5]
    return images, boxes


# Instanciar para verificacion de que los archivos son accesibles
ds_check = DentexDataset('train', YOLO_G8b)
log(f'DentexDataset train: {len(ds_check)} imgs', 'OK')
log(f'  class_counts: {dict(ds_check.class_counts)}', 'DATA')

# Mostrar estructura de un sample
img_t, boxes_t = ds_check[0]
log(f'  Muestra [0]: img={tuple(img_t.shape)}  boxes={tuple(boxes_t.shape)}', 'DATA')
log(f'  boxes[0]: {boxes_t[0].tolist() if len(boxes_t) else "(sin boxes)"}', 'DATA')


---

## b) Particionado

> El particionado fue realizado en el **Notebook 01** (función `build_g8b_dataset`). Los archivos ya están en disco con `seed=42`. Este notebook los lee tal cual — no los regenera.

Los números de referencia (leídos en la celda siguiente desde disco) son:

| Split | Imágenes | % total | Criterio |
|-------|----------|---------|----------|
| train | ~127 | ~70% | Maximizar data de entrenamiento |
| val | ~27 | ~15% | Early stopping y ajuste de hiperparámetros |
| test | ~27 | ~15% | Evaluación final — nunca visto durante entrenamiento |

**Seed:** `42` — garantiza la misma partición en cualquier ejecución.  
**Nota:** con una sola clase (`impacted`) no aplica estratificación por clase — el shuffle con seed fija es suficiente para reproducibilidad.


In [ ]:
# ── Reportar distribución real de los splits desde disco ─────────────────
split_stats = {}

for split in ['train', 'val', 'test']:
    img_dir = YOLO_G8b / 'images' / split
    lbl_dir = YOLO_G8b / 'labels' / split
    if not img_dir.exists():
        log(f'{split}: directorio no encontrado — ejecutar notebook 01', 'ERR')
        continue
    imgs    = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    n_boxes = sum(
        len([l for l in lbl.read_text().strip().split('\n') if l.strip()])
        for lbl in lbl_dir.glob('*.txt')
    )
    split_stats[split] = {'n_imgs': len(imgs), 'n_boxes': n_boxes}

total_imgs  = sum(v['n_imgs']  for v in split_stats.values())
total_boxes = sum(v['n_boxes'] for v in split_stats.values())

log('Distribucion de splits — yolo_g8b_small (generado en notebook 01):', 'DATA')
print()
print(f'{"Split":8s}  {"Imagenes":10s}  {"% total":8s}  {"Boxes impacted":16s}')
print('-' * 52)
for split, s in split_stats.items():
    pct = s['n_imgs'] / max(total_imgs, 1) * 100
    print(f'{split:8s}  {s["n_imgs"]:10d}  {pct:7.1f}%  {s["n_boxes"]:16d}')
print('-' * 52)
print(f'{"TOTAL":8s}  {total_imgs:10d}  {"100.0%":8s}  {total_boxes:16d}')


---

## c) Preprocesamiento

> La justificación del pipeline (por qué 640px, por qué ImageNet stats, manejo de escala de grises) está en el **Notebook 01, Sección e)**.

Acá se definen las constantes y el `BASE_TRANSFORM` que usan todos los DataLoaders. Val y test **siempre** usan `BASE_TRANSFORM` — sin ninguna transformación adicional.


In [ ]:
MEAN     = [0.485, 0.456, 0.406]
STD      = [0.229, 0.224, 0.225]
IMG_SIZE = 640

BASE_TRANSFORM = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

MEAN_T = torch.tensor(MEAN).view(3, 1, 1)
STD_T  = torch.tensor(STD).view(3, 1, 1)

def desnormalizar(tensor):
    return torch.clamp(tensor * STD_T + MEAN_T, 0.0, 1.0)


# Verificar rango sobre una imagen real
sample_imgs = list((YOLO_G8b/'images'/'train').glob('*.jpg'))
if sample_imgs:
    real = Path(os.path.realpath(str(sample_imgs[0])))
    t    = BASE_TRANSFORM(Image.open(real).convert('RGB'))
    log('BASE_TRANSFORM verificado:', 'OK')
    log(f'  shape: {tuple(t.shape)}  (C x H x W)', 'DATA')
    log(f'  dtype: {t.dtype}', 'DATA')
    log(f'  min:   {t.min():.4f}  max: {t.max():.4f}', 'DATA')
    log(f'  mean:  {t.mean():.4f}  std: {t.std():.4f}', 'DATA')
else:
    log('No se encontraron imagenes — ejecutar notebook 01', 'ERR')


---

## d) Data augmentation

### Augmentación interna de YOLOv8 (usada en todos los experimentos)

YOLOv8 aplica augmentación **internamente durante el entrenamiento**, sincronizando cada transformación geométrica con los bounding boxes automáticamente. Esta es la única estrategia que se usó — en todos los experimentos del proyecto.

Los parámetros pasados a `model.train()` en el experimento de producción (G8b) son:

| Parámetro | Valor | Efecto |
|-----------|-------|--------|
| `fliplr` | 0.5 | Flip horizontal con p=50% — las muelas aparecen en ambos lados del arco |
| `hsv_v` | 0.4 | Variación de brillo ±40% — simula diferencias de exposición entre equipos de RX |
| `hsv_s` | 0.4 | Variación de saturación — afecta contraste al convertir a RGB |
| `degrees` | 5.0 | Rotación ±5° — simula leve inclinación al posicionar al paciente |
| `translate` | 0.1 | Traslación ±10% — variación en el encuadre de la panorámica |
| `mosaic` | 0.0 | **Desactivado** — combinar panorámicas de distintos pacientes no tiene sentido clínico |
| `mixup` | 0.0 | **Desactivado** — mezclar radiografías genera artefactos sin valor |

### ¿Por qué no se implementaron augmentaciones con `torchvision`?

Para detección de objetos, la augmentación geométrica (flip, rotación, traslación) debe aplicarse **simultáneamente** a la imagen y a todos los bounding boxes. `torchvision.transforms` opera solo sobre la imagen — no propaga las transformaciones a las coordenadas de los boxes.

| | `torchvision` | YOLOv8 interno |
|-|---------------|----------------|
| Flip / rotación / traslación | ✓ imagen sola | ✓ imagen + boxes |
| Ajuste de brillo y contraste | ✓ | ✓ (HSV) |
| **Coordenadas de bounding boxes** | **✗** | **✓ nativo** |

> El `BASE_TRANSFORM` definido en la sección c) (resize + normalizar) se usa para val y test. Para train, YOLOv8 aplica su propio pipeline que incluye resize, normalización y las augmentaciones de la tabla de arriba — todo internamente.


In [ ]:
# Parámetros de augmentación G8b — se pasan a model.train() en el notebook de entrenamiento
AUG_G8b = dict(
    fliplr    = 0.5,
    hsv_v     = 0.4,
    hsv_s     = 0.4,
    degrees   = 5.0,
    translate = 0.1,
    mosaic    = 0.0,
    mixup     = 0.0,
)

# Val y test usan BASE_TRANSFORM (solo resize + normalizar, sin augmentación)
# Train: YOLOv8 maneja su propio pipeline con AUG_G8b internamente
log('AUG_G8b definido — se pasa a model.train() en el notebook de entrenamiento', 'OK')
for k, v in AUG_G8b.items():
    log(f'  {k}: {v}', 'DATA')


> **Nota:** no se incluye visualización de augmentaciones con `torchvision` porque esas transformaciones nunca se usaron. Las augmentaciones reales las aplica YOLOv8 internamente durante el entrenamiento y son visibles en las curvas de entrenamiento del notebook de experimentos.


### Instanciación de los DataLoaders

Los DataLoaders se instancian con:

| Split | Transform | shuffle | Justificación |
|-------|-----------|---------|---------------|
| train | `F_mixaug` (mayor intensidad) | `True` | Shuffle evita que el modelo aprenda el orden del dataset |
| val | `BASE_TRANSFORM` | `False` | Eval determinista — mismos resultados en cada epoch |
| test | `BASE_TRANSFORM` | `False` | Eval final determinista |
| verify | `A_baseline` | `True` | Solo para verificación visual — sin transformaciones geométricas que desplacen las boxes |


In [ ]:
# ── DataLoaders ──────────────────────────────────────────────────────────

# Train — augmentation maxima (F_mixaug)
ds_train = DentexDataset('train', YOLO_G8b, transform=AUGMENTATIONS['F_mixaug'])
dl_train = DataLoader(
    ds_train,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    num_workers = 2,
    collate_fn  = collate_fn,
    pin_memory  = torch.cuda.is_available(),
    generator   = torch.Generator().manual_seed(SEED),
)

# Val — sin augmentation
ds_val  = DentexDataset('val',  YOLO_G8b, transform=BASE_TRANSFORM)
dl_val  = DataLoader(ds_val,  batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=2, collate_fn=collate_fn)

# Test — sin augmentation
ds_test = DentexDataset('test', YOLO_G8b, transform=BASE_TRANSFORM)
dl_test = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=2, collate_fn=collate_fn)

# Verificacion visual — A_baseline para que las boxes cuadren con los pixeles
ds_verify = DentexDataset('train', YOLO_G8b, transform=AUGMENTATIONS['A_baseline'])
dl_verify = DataLoader(ds_verify, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=0, collate_fn=collate_fn,
                       generator=torch.Generator().manual_seed(SEED))

log('DataLoaders instanciados:', 'OK')
log(f'  dl_train: {len(ds_train)} imgs  |  {len(dl_train)} batches  |  aug=F_mixaug  shuffle=True', 'DATA')
log(f'  dl_val:   {len(ds_val)}  imgs  |  {len(dl_val)}  batches  |  aug=BASE     shuffle=False', 'DATA')
log(f'  dl_test:  {len(ds_test)} imgs  |  {len(dl_test)} batches  |  aug=BASE     shuffle=False', 'DATA')
log(f'  dl_verify:{len(ds_verify)} imgs  |  sin aug  |  para visualizacion', 'DATA')

# Confirmacion explicita: train usa F_mixaug; val/test usan BASE
assert ds_train.transform is not BASE_TRANSFORM, 'ERROR: train no deberia usar BASE_TRANSFORM'
assert ds_val.transform   is BASE_TRANSFORM,     'ERROR: val deberia usar BASE_TRANSFORM'
assert ds_test.transform  is BASE_TRANSFORM,     'ERROR: test deberia usar BASE_TRANSFORM'
log('Assert OK — augmentation correctamente separada de val/test', 'OK')


---

## e) Verificación final

Se extrae un batch del DataLoader de train y se verifican:
1. Dimensiones de los tensores
2. Rango de valores tras la normalización
3. Visualización con imágenes desnormalizadas y bounding boxes superpuestas
4. Correcta correspondencia imagen–etiqueta


In [ ]:
# ── 1. Extraer batch y verificar dimensiones ─────────────────────────────
torch.manual_seed(SEED)
imgs_batch, boxes_batch = next(iter(dl_verify))

log('=== Verificacion del batch ===', 'DATA')
log(f'  imgs_batch.shape:  {tuple(imgs_batch.shape)}'  # (B, C, H, W)
    f'  — (batch_size={imgs_batch.shape[0]}, canales={imgs_batch.shape[1]}, '
    f'alto={imgs_batch.shape[2]}, ancho={imgs_batch.shape[3]})', 'DATA')
log(f'  imgs_batch.dtype:  {imgs_batch.dtype}', 'DATA')
log(f'  imgs_batch.min:    {imgs_batch.min():.4f}', 'DATA')
log(f'  imgs_batch.max:    {imgs_batch.max():.4f}', 'DATA')
log(f'  imgs_batch.mean:   {imgs_batch.mean():.4f}', 'DATA')
log(f'  imgs_batch.std:    {imgs_batch.std():.4f}', 'DATA')
log('', 'INFO')
log('  boxes por imagen en este batch:', 'DATA')
for i, bx in enumerate(boxes_batch):
    log(f'    img[{i}]: {len(bx)} boxes — {bx.tolist() if len(bx) else "(ninguna)"}', 'DATA')


In [ ]:
# ── 2. Visualizacion del batch — imagenes desnormalizadas + boxes ─────────

def dibujar_boxes_yolo(ax, img_tensor, boxes_tensor, title=''):
    """
    Muestra img_tensor desnormalizado con las bounding boxes YOLO superpuestas.
    boxes_tensor: [N, 5] con [cls, cx, cy, w, h] normalizados [0,1].
    """
    vis = desnormalizar(img_tensor).permute(1, 2, 0).numpy()
    H, W = vis.shape[:2]
    ax.imshow(vis)

    for box in boxes_tensor:
        cls, cx, cy, bw, bh = box.tolist()
        x1 = (cx - bw/2) * W; y1 = (cy - bh/2) * H
        bw_px = bw * W;       bh_px = bh * H
        color = '#FF4444'
        rect = patches.Rectangle((x1, y1), bw_px, bh_px,
                                   linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1 + 2, y1 + 14, f'imp {bw_px:.0f}x{bh_px:.0f}px',
                fontsize=7, color='white',
                bbox=dict(boxstyle='round,pad=0.1', fc=color, alpha=0.8))

    n_boxes = len(boxes_tensor)
    ax.set_title(
        f'{title}\n{W}x{H}px | {n_boxes} box{"es" if n_boxes != 1 else ""}',
        fontsize=8, fontweight='bold')
    ax.axis('off')


n_show = min(BATCH_SIZE, len(boxes_batch))
cols = 4; rows = (n_show + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 4 * rows))
axes = np.array(axes).flatten()

for i in range(n_show):
    dibujar_boxes_yolo(
        axes[i],
        imgs_batch[i],
        boxes_batch[i],
        title=f'train[{i}]'
    )

for i in range(n_show, len(axes)):
    axes[i].set_visible(False)

plt.suptitle(
    f'Batch de train — imagenes desnormalizadas con bounding boxes\n'
    f'shape={tuple(imgs_batch.shape)}  |  rango post-norm: [{imgs_batch.min():.2f}, {imgs_batch.max():.2f}]',
    fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUTS_DIR / 'batch_verification.png'), dpi=120, bbox_inches='tight')
plt.show()
log('Verificacion visual del batch guardada en data/outputs/batch_verification.png', 'OK')


In [ ]:
# ── 3. Verificacion cruzada imagen-etiqueta ───────────────────────────────
# Comprobar que cada imagen tiene al menos 1 box en el dataset (invariante del G8b)

log('Verificacion cruzada imagen-etiqueta:', 'DATA')
issues = []
for split in ['train', 'val', 'test']:
    ds = DentexDataset(split, YOLO_G8b, transform=BASE_TRANSFORM)
    n_ok = n_empty = 0
    for img_t, boxes_t in ds:
        if len(boxes_t) == 0:
            n_empty += 1
        else:
            n_ok += 1
            # Verificar que coordenadas estan en [0,1]
            assert boxes_t[:, 1:].min() >= 0.0, f'Coordenada negativa en {split}'
            assert boxes_t[:, 1:].max() <= 1.0, f'Coordenada > 1 en {split}'
    log(f'  {split:5s}: {n_ok} OK  |  {n_empty} sin boxes', 'DATA')
    if n_empty > 0:
        issues.append(f'{split}: {n_empty} imagenes sin boxes')

if issues:
    log(f'ADVERTENCIA: {issues}', 'WARN')
else:
    log('Todas las imagenes tienen al menos 1 box — OK', 'OK')

# ── 4. Resumen final ─────────────────────────────────────────────────────
log('', 'INFO')
log('=== RESUMEN FINAL ===', 'DATA')
log(f'Dataset:      yolo_g8b_small  |  nc=1 (solo impacted)', 'DATA')
log(f'Input size:   {IMG_SIZE}x{IMG_SIZE}px  |  3 canales (RGB)', 'DATA')
log(f'Norm:         mean={MEAN}', 'DATA')
log(f'              std ={STD}', 'DATA')
log(f'Batch shape:  {tuple(imgs_batch.shape)}  (B x C x H x W)', 'DATA')
log(f'Val/Test aug: BASE_TRANSFORM (sin augmentation)', 'DATA')
log(f'Train aug:    F_mixaug (flip + jitter + blur + erasing)', 'DATA')
log(f'Seed:         {SEED}  (reproducibilidad garantizada)', 'DATA')


---

## Resumen del notebook

| Sección | Contenido propio de este notebook |
|---------|-----------------------------------|
| **a)** | Clase `DentexDataset` (PyTorch `Dataset`) con `__len__`/`__getitem__`/`collate_fn`. `DentexDetectionDataset` para visualización. Justificación de por qué no aplica `ImageFolder`. |
| **b)** | Referencia al particionado del Notebook 01. Tabla con los números reales leídos desde disco. |
| **c)** | Definición de `BASE_TRANSFORM` y `desnormalizar()`. Verificación del rango post-normalización. |
| **d)** | Explicación de por qué se usan las augmentaciones internas de YOLOv8. Tabla de parámetros G8b. Justificación de por qué `torchvision` no es adecuado para detección. |
| **e)** | DataLoaders (`dl_train`, `dl_val`, `dl_test`) con `shuffle`, `num_workers`, `pin_memory`. Batch visual con boxes desnormalizadas. Verificación cruzada imagen–etiqueta en los 3 splits. |


